# CRISP-DM Masterclass: Online Retail

**Goal:** build one coherent data-science solution from business question to final recommendation.

The six CRISP-DM phases are:
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment / Synthesis

For every technique, ask:
1. What business question does it answer?
2. What representation of the data does it require?
3. What assumptions does it make?
4. What decision could change because of the result?

# Phase 1 — Business Understanding

### Scenario
You are the lead data scientist for an online giftware retailer. Leadership wants a customer-intelligence system supporting retention, merchandising, and operational investigation.

### Objectives
- Segment customers by behavior.
- Detect unusual customer behavior.
- Predict near-term repeat purchase.
- Discover product affinities.
- Build approximate product-similarity search.

### Quiz 1 — Business framing

**Q1. Which is strongest?**

A. Run K-Means.  
B. Find interesting patterns.  
C. Identify customer segments that support differentiated retention actions and can be described using measurable behavioral features.  
D. Make a dashboard.

**Answer:** C.

**Q2. Why should the business question come before the algorithm?**

**Answer:** the same data can support many valid models, but only some solve the actual decision problem.

# Phase 2 — Data Understanding

Inspect the raw data **before cleaning it**. This preserves an audit trail.

### Quiz 2

**Q1. Why inspect cancellations before removing them?**

Because cancellations are information about the business process and may affect revenue interpretation.

**Q2. A missing CustomerID means:**

A. The customer did not exist.  
B. The row is automatically invalid.  
C. We cannot safely attribute the transaction to a known customer.  
D. The customer spent zero.

**Answer:** C.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW = Path("../data/raw")
xlsx_candidates = list(RAW.glob("*.xlsx"))
xlsx_candidates

In [ ]:
from src.data_prep import load_raw
assert xlsx_candidates, "Place Online Retail.xlsx in data/raw/ before running."
raw = load_raw(xlsx_candidates[0])
raw.shape, raw.head()

In [ ]:
raw.info()
raw.isna().mean().sort_values(ascending=False)

In [ ]:
raw.describe(include="all").T

In [ ]:
audit = pd.Series({
    "rows": len(raw),
    "duplicate_rows": raw.duplicated().sum(),
    "missing_customer_id": raw["CustomerID"].isna().sum(),
    "cancellation_rows": raw["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "non_positive_quantity": (raw["Quantity"] <= 0).sum(),
    "non_positive_unit_price": (raw["UnitPrice"] <= 0).sum(),
    "invalid_dates": raw["InvoiceDate"].isna().sum(),
})
audit

In [ ]:
raw["line_revenue"] = raw["Quantity"] * raw["UnitPrice"]
raw["month"] = raw["InvoiceDate"].dt.to_period("M").astype(str)
monthly = raw.groupby("month")["line_revenue"].sum()
monthly.plot(figsize=(11,4), title="Raw monthly line revenue")
plt.ylabel("GBP")
plt.tight_layout()
plt.show()

### Interpretation checkpoint

Do not interpret the raw revenue plot as final "sales" yet. It mixes cancellations, missing customer IDs, and potentially invalid records.

**Quiz 3:** Which statement is best?

A. The raw data is wrong.  
B. Every dataset must be perfect.  
C. We need an explicit analytical definition of a valid sale before computing business KPIs.

**Answer:** C.

# Phase 3 — Data Preparation

We create an analytical sales table using explicit rules:

1. Preserve cancellations in the raw audit trail.
2. Exclude cancellations from positive-sales customer modeling.
3. Require known CustomerID for customer-level analysis.
4. Require positive quantity and unit price.
5. Parse dates.
6. Engineer line revenue.

These are documented assumptions for this business question, not universal truths.

### Quiz 4 — Feature engineering

Why is customer-level aggregation preferable to clustering transaction rows for a customer-segmentation question?

**Answer:** the business entity being segmented is the customer. Transaction-level clustering can produce purchase archetypes rather than customer archetypes.

In [ ]:
from src.data_prep import clean_sales, build_rfm, build_repeat_purchase_dataset
sales = clean_sales(raw)
sales.shape, sales.head()

In [ ]:
cleaning_summary = pd.Series({
    "raw_rows": len(raw),
    "analytical_sales_rows": len(sales),
    "rows_removed": len(raw) - len(sales),
    "unique_customers": sales["CustomerID"].nunique(),
    "unique_invoices": sales["InvoiceNo"].nunique(),
    "unique_products": sales["StockCode"].nunique(),
})
cleaning_summary

In [ ]:
rfm = build_rfm(sales)
rfm.describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ["recency","frequency","monetary"]):
    sns.histplot(np.log1p(rfm[col]), kde=True, ax=ax)
    ax.set_title(f"log1p({col})")
plt.tight_layout()
plt.show()

### Why log transforms?

Frequency and monetary value are usually right-skewed. K-Means is distance-based, so a few extreme customers can dominate.

**Quiz 5:** What does standardization solve that log transformation does not?

- Log transformation reduces skew.
- Standardization puts variables on comparable scale.

They solve different problems.

# Phase 4A — Unsupervised Learning: Customer Clustering

K-Means is a transparent baseline after transformation and scaling. It does not discover the "true" number of customer types, so `k` should be selected using both quantitative and business criteria.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X = np.log1p(rfm[["recency","frequency","monetary"]])
X_scaled = StandardScaler().fit_transform(X)

scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

pd.Series(scores, name="silhouette")

In [ ]:
best_k = max(scores, key=scores.get)
best_k

In [ ]:
from src.modeling import rfm_cluster
cluster_model, clustered, silhouette = rfm_cluster(rfm, k=best_k)
print("Silhouette:", silhouette)
segment_profile = clustered.groupby("cluster").agg(
    customers=("CustomerID","count"),
    recency=("recency","median"),
    frequency=("frequency","median"),
    monetary=("monetary","median"),
    revenue=("monetary","sum")
).sort_values("revenue", ascending=False)
segment_profile

### Quiz 6 — Clustering evaluation

**Q1. A high silhouette score proves clusters are commercially useful.**

**False.** It supports geometric separation, not business usefulness.

**Q2. If one cluster has the highest revenue, should every customer in it receive the same treatment?**

Not necessarily. Segments are abstractions and can contain meaningful within-segment differences.

# Phase 4B — Anomaly / Outlier Detection

Isolation Forest ranks unusual customer behavior.

Possible interpretations include unusually high-value customers, one-off wholesale-like behavior, unusual purchase frequency, data-quality problems, or cases worth investigation.

**Anomaly ≠ fraud.**

In [ ]:
from src.modeling import isolation_forest
if_model, anomalies = isolation_forest(rfm, contamination=0.03)
anomalies.sort_values("anomaly_score", ascending=False).head(15)

In [ ]:
anomaly_summary = anomalies.groupby("anomaly").agg(
    customers=("CustomerID","count"),
    median_recency=("recency","median"),
    median_frequency=("frequency","median"),
    median_monetary=("monetary","median")
)
anomaly_summary

### Quiz 7 — Anomaly detection

**Q1. A flagged customer is fraudulent.** False.

**Q2. Why rank anomaly scores?** Because operations have limited investigation capacity.

**Q3. Main limitation?** There is often no ground-truth label, so expert review or later outcomes are needed for validation.

# Phase 4C — Supervised Learning: Repeat Purchase

### Prediction problem
At a historical cutoff date, predict whether a customer purchases in the following 30 days.

**Target:** `repeat_30d`

**Features:** only information available before the cutoff.

### Leakage warning
Random train/test splitting is inappropriate for this forecasting setup. We use temporal cutoffs.

In [ ]:
cutoff_train = sales["InvoiceDate"].max() - pd.Timedelta(days=90)
cutoff_test = sales["InvoiceDate"].max() - pd.Timedelta(days=45)

train_ds = build_repeat_purchase_dataset(sales, cutoff=cutoff_train, horizon_days=30, lookback_days=180)
test_ds = build_repeat_purchase_dataset(sales, cutoff=cutoff_test, horizon_days=30, lookback_days=180)

print(train_ds.shape, test_ds.shape)
print("Train positive rate:", train_ds["repeat_30d"].mean())
print("Test positive rate:", test_ds["repeat_30d"].mean())

In [ ]:
from src.modeling import temporal_classifier
results = temporal_classifier(train_ds, test_ds)

pd.DataFrame({
    name: {"ROC-AUC": r["roc_auc"], "PR-AUC": r["pr_auc"]}
    for name, r in results.items()
}).T

### Quiz 8 — Classification

**Q1. Is accuracy enough with class imbalance?** Usually no.

Useful metrics include ROC-AUC, PR-AUC, precision, recall, calibration, and business value at a chosen threshold.

**Q2. Why is PR-AUC useful?** It focuses on the positive-class precision/recall trade-off.

**Q3. What is the operational decision?** Which customers should receive an intervention under a budget constraint—not simply which algorithm has the highest score.

In [ ]:
from sklearn.metrics import precision_score, recall_score

best_name = max(results, key=lambda n: results[n]["pr_auc"])
p = results[best_name]["predicted_probability"]
y = test_ds["repeat_30d"].to_numpy()

rows = []
for t in np.arange(0.10, 0.91, 0.05):
    pred = (p >= t).astype(int)
    rows.append({
        "threshold": round(float(t),2),
        "precision": precision_score(y,pred,zero_division=0),
        "recall": recall_score(y,pred,zero_division=0),
        "customers_targeted": int(pred.sum())
    })
pd.DataFrame(rows)

# Phase 4D — Associative Rule Mining

For market-basket analysis:
- one row = invoice
- columns = products
- values = whether the product appears

Metrics:
- **Support:** fraction of baskets containing an itemset.
- **Confidence:** P(B | A).
- **Lift:** confidence divided by P(B).

Lift > 1 suggests positive association relative to independence, but **does not prove causation**.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

top_products = sales.groupby("StockCode")["line_revenue"].sum().nlargest(80).index
basket = (
    sales[sales["StockCode"].isin(top_products)]
    .assign(value=1)
    .pivot_table(index="InvoiceNo", columns="StockCode", values="value", aggfunc="max", fill_value=0)
    .astype(bool)
)
print(basket.shape)

freq = apriori(basket, min_support=0.02, use_colnames=True, max_len=3)
rules = association_rules(freq, metric="lift", min_threshold=1.2).sort_values(["lift","confidence"], ascending=False)
rules[["antecedents","consequents","support","confidence","lift"]].head(15)

### Quiz 9 — Association rules

**Q1. 90% confidence means A causes B.** False.

**Q2. Why can confidence mislead?** A very common consequent can create high confidence without a strong relationship.

**Q3. Why use lift?** It compares observed co-occurrence with the rate expected under independence.

# Phase 4E — Sub-linear Search with LSH

Suppose the product catalog becomes very large. Exhaustively comparing every product with every other product is quadratic in catalog size.

LSH uses hashing to retrieve candidate neighbors without comparing every pair.

### MinHash intuition

For set representations, Jaccard similarity is:

`J(A,B) = |A ∩ B| / |A ∪ B|`

MinHash estimates Jaccard similarity. LSH groups similar signatures into buckets, then retrieves candidates from matching buckets.

It is approximate: there can be false negatives and false-positive candidates.

In [ ]:
from src.lsh import MinHashLSH, shingles, jaccard

products = (
    sales[["StockCode","Description"]]
    .dropna()
    .drop_duplicates("StockCode")
    .head(1500)
)
token_sets = {}

lsh = MinHashLSH(num_perm=64, bands=16)
for row in products.itertuples(index=False):
    tokens = shingles(row.Description)
    token_sets[row.StockCode] = tokens
    lsh.add(row.StockCode, tokens)

query_id = products.iloc[0]["StockCode"]
candidates = lsh.query(token_sets[query_id])

scored = []
for candidate in candidates:
    if candidate == query_id:
        continue
    desc = products.loc[products["StockCode"] == candidate, "Description"].iloc[0]
    scored.append({"StockCode":candidate, "Description":desc, "jaccard":jaccard(token_sets[query_id], token_sets[candidate])})

pd.DataFrame(scored).sort_values("jaccard", ascending=False).head(10)

### Quiz 10 — LSH

**Q1. LSH guarantees the exact nearest neighbor.** False.

**Q2. Why is it useful at scale?** Hash buckets can drastically reduce candidate comparisons relative to exhaustive pairwise search.

**Q3. What controls the trade-off?** Number of permutations, bands, rows per band, tokenization, and similarity threshold.

# Phase 5 — Evaluation: Bring the Models Together

CRISP-DM evaluation is broader than selecting the highest offline metric.

| Output | Quantitative evaluation | Business evaluation |
|---|---|---|
| Clusters | silhouette / stability | interpretability and actionability |
| Anomalies | investigation yield | operational usefulness |
| Classifier | ROC-AUC / PR-AUC / precision / recall | campaign value at capacity |
| Rules | support / confidence / lift | merchandising plausibility |
| LSH | candidate recall / similarity | retrieval usefulness and latency |

### Quiz 11

**The best model is always the one with the highest offline metric.**

**False.** A production choice balances quality, stability, interpretability, cost, latency, maintainability, and business value.

# Phase 6 — Deployment / Synthesis

## Recommended weekly operating model

1. Ingest transactions.
2. Validate schema and data quality.
3. Recompute RFM.
4. Assign customers to segments.
5. Score anomalies.
6. Score repeat-purchase propensity.
7. Refresh association rules.
8. Update the product-similarity index.
9. Monitor drift and business KPIs.

## Monitoring

**Data:** missing CustomerID rate, cancellation rate, negative quantity rate, price distribution, transaction volume.

**Models:** score distribution, segment proportions, anomaly volume, delayed-label PR-AUC, feature drift.

**Business:** repeat-purchase rate, campaign conversion, revenue per targeted customer, investigation yield.

### Final quiz

**Q1. Is CRISP-DM one-way?** No. Evaluation often sends you back to earlier phases.

**Q2. Is a highly accurate model that cannot support a stakeholder decision finished?** No.

**Q3. What is the final artifact?**

A defensible chain from **business decision → data → method → evaluation → action → monitoring**.

## Completion checklist

- [ ] Business problem and success criteria
- [ ] Data provenance
- [ ] Raw-data audit
- [ ] Cleaning rules
- [ ] RFM feature engineering
- [ ] Customer clustering
- [ ] Anomaly detection
- [ ] Temporal repeat-purchase model
- [ ] Association rules
- [ ] LSH retrieval
- [ ] Metrics interpreted
- [ ] Limitations documented
- [ ] Deployment/monitoring plan
- [ ] Final recommendation